In [1]:
import numpy as np
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU
from tensorflow.keras.optimizers import Adam

# Load mappings
with open("char_to_num.pkl", "rb") as f:
    char_to_num = pickle.load(f)

with open("num_to_char.pkl", "rb") as f:
    num_to_char = pickle.load(f)

# Load training data
X = np.load("X.npy")
y = np.load("y.npy")

sequence_length = 40
vocab_size = len(char_to_num)

# Build GRU model
model = Sequential()
model.add(GRU(128, input_shape=(sequence_length, vocab_size)))
model.add(Dense(vocab_size, activation='softmax'))

# Compile
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001))

model.summary()


C:\Users\chala\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 128)            │        74,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,265 (325.25 KB)

 Trainable params: 83,265 (325.25 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
history = model.fit(X, y, batch_size=128, epochs=2)


Epoch 1/2
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 426s 49ms/step - loss: 1.9397
Epoch 2/2
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 546s 63ms/step - loss: 1.5983


In [4]:
# load the saved dataset text
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# the same window length we used before
sequence_length = 40

import pickle, numpy as np

with open("char_to_num.pkl", "rb") as f:
    char_to_num = pickle.load(f)
with open("num_to_char.pkl", "rb") as f:
    num_to_char = pickle.load(f)

# fast load of training tensors
X = np.load("X.npy")
y = np.load("y.npy")

vocab_size = len(char_to_num)


import numpy as np

def generate_text(model, seed_text, length=300, temperature=0.7):
    generated = seed_text
    for _ in range(length):
        x = np.zeros((1, sequence_length, vocab_size), dtype=np.float32)
        for t, ch in enumerate(seed_text):
            if ch in char_to_num:
                x[0, t, char_to_num[ch]] = 1.0

        preds = model.predict(x, verbose=0)[0]
        # temperature sampling
        preds = np.log(preds + 1e-8) / temperature
        probs = np.exp(preds) / np.sum(np.exp(preds))
        next_ch = np.random.choice(list(char_to_num.keys()), p=probs)

        generated += next_ch
        seed_text = seed_text[1:] + next_ch
    return generated


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam

model = Sequential([
    GRU(128, input_shape=(sequence_length, vocab_size)),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=Adam(1e-3))
model.summary()

# quick compare run: 1–2 epochs is enough
history = model.fit(X, y, batch_size=128, epochs=2)

seed = text[:sequence_length]

print("temp=0.3\n", generate_text(model, seed, length=400, temperature=0.3))
print("\ntemp=0.7\n", generate_text(model, seed, length=400, temperature=0.7))
print("\ntemp=1.2\n", generate_text(model, seed, length=400, temperature=1.2))



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 128)            │        74,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,265 (325.25 KB)

 Trainable params: 83,265 (325.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 315s 36ms/step - loss: 1.9456
Epoch 2/2
8714/8714 ━━━━━━━━━━━━━━━━━━━━ 348s 40ms/step - loss: 1.5968
temp=0.3
 First Citizen:
Before we proceed any further the proveding
And that the dead the maid it the fair the could the tongue,
And the streng the pristion with my lord.

BRUTUS:
I will see the dead the man the must thee?

LADY CAPULET:
God say the state of my langed the mand the sight
And the disposed the fight the mand the state
And made the man the death the state the courted
To my made the dead the served the senter
To see the must the self

temp=0.7
 First Citizen:
Before we proceed any further'd much their your fail
To see sholeny queen the worth man, thy deed, he with their
Madser that I here and shall thee the hisse.

AUFIDIUS:
It made not stoul'd in his good for mercy,
With the tegrimant you may most been in the

CLord German:
Well all the content to enticution to and
At therefore which he with him as coulted my crown,
What it contury 

In [5]:
model.save("gru_textgen.h5")
